In [7]:
!pip install pandas
import sqlite3
import pandas as pd
import os


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
db_path = r"C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\shopmart-ecommerce-analytics\data\shopmart.db"
raw_path = r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\shopmart-ecommerce-analytics\data\raw'
os.makedirs(r"C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\shopmart-ecommerce-analytics\data\processed", exist_ok = True)

def get_connection():
    """return a sqlite connection with performance settings enabled"""
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA journal_mode = WAL")    # faster writes
    conn.execute("PRAGMA synchronous  = NORMAL") # safe + fast
    conn.execute("PRAGMA cache_size   = -64000") # 64MB cache
    conn.execute("PRAGMA foreign_keys = ON")     # enforce FK rules
    return conn

def create_tables(conn):
    """Drop and recreate all 8 tables with correct schema."""
    cursor = conn.cursor()

    """Drop in child-first order to respect foreign keys"""
    for t in ["order_items","payments","reviews",
          "orders","customers","products",
          "sellers","category_translation"]:
        print(f"Dropping table: {t}")
        cursor.execute(f"DROP TABLE IF EXISTS {t}")
    
    cursor.executescript("""
        CREATE TABLE customers(
            customer_id         text primary key,
            customer_unique_id  text not null,
            customer_zip_code   text,
            customer_city       text,
            customer_state      text
        );
        
        CREATE TABLE orders (
            order_id                      TEXT PRIMARY KEY,
            customer_id                   TEXT NOT NULL,
            order_status                  TEXT,
            order_purchase_timestamp      TEXT,
            order_approved_at             TEXT,
            order_delivered_carrier_date  TEXT,
            order_delivered_customer_date TEXT,
            order_estimated_delivery_date TEXT,
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );
                         
        CREATE TABLE order_items (
            order_id            TEXT,
            order_item_id       INTEGER,
            product_id          TEXT,
            seller_id           TEXT,
            shipping_limit_date TEXT,
            price               REAL,
            freight_value       REAL,
            PRIMARY KEY (order_id, order_item_id)
        );
        
        CREATE TABLE payments (
            order_id             TEXT,
            payment_sequential   INTEGER,
            payment_type         TEXT,
            payment_installments INTEGER,
            payment_value        REAL
        );
        
        CREATE TABLE products (
            product_id                  TEXT PRIMARY KEY,
            product_category_name       TEXT,
            product_name_length         INTEGER,
            product_description_length  INTEGER,
            product_photos_qty          INTEGER,
            product_weight_g            INTEGER,
            product_length_cm           INTEGER,
            product_height_cm           INTEGER,
            product_width_cm            INTEGER
        );

        CREATE TABLE reviews (
            review_id               TEXT,
            order_id                TEXT,
            review_score            INTEGER,
            review_comment_title    TEXT,
            review_comment_message  TEXT,
            review_creation_date    TEXT,
            review_answer_timestamp TEXT
        );
                         
        CREATE TABLE sellers (
            seller_id       TEXT PRIMARY KEY,
            seller_zip_code TEXT,
            seller_city     TEXT,
            seller_state    TEXT
        );

        CREATE TABLE category_translation (
            category_portuguese TEXT PRIMARY KEY,
            category_english    TEXT
        );
    """)
    
    # Indexes: make JOINs and WHERE clauses fast
    cursor.executescript("""
        create index idx_orders_customer on orders(customer_id);
        create index idx_order_status on orders(order_status);
        create index idx_orders_date on orders(order_purchase_timestamp);
        create index idx_items_order on order_items(order_id);
        create index idx_items_product on order_items(product_id);
        create index idx_payments_order on payments(order_id);
        create index idx_reviews_order on reviews(order_id);
        create index idx_cust_unique on customers(customer_unique_id);
    """)
    conn.commit()
    print('✅ Tables and indexes created')


def import_csv(conn, filename, table, rename_cols = None):
    """Load one CSV file into one sqlite table"""
    path = os.path.join(raw_path, filename)
    if not os.path.exists(path):
        print(f" Missing {filename} -> place it in {raw_path}")
        return 0
    df = pd.read_csv(path, low_memory = False)
    if rename_cols:
        df = df.rename(columns = rename_cols)
    df.to_sql(table, conn, if_exists = 'replace', index = False, chunksize = 5000)
    n = conn.execute(
        f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print('✅ {table : <40} {n:>9} rows')
    return n

def import_all_csvs(conn):
    import_csv(conn, 'olist_customers.csv', 'customers')
    import_csv(conn, 'olist_orders.csv', 'orders')
    import_csv(conn, "olist_order_items.csv", "order_items")
    import_csv(conn, "olist_order_payments.csv", "payments")
    import_csv(conn, "olist_products.csv", "products")
    import_csv(conn, "olist_order_reviews.csv", "reviews")
    import_csv(conn, "olist_sellers.csv", "sellers")
    import_csv(conn, "product_category_name_translation.csv", "category_translation", rename_cols = {"product_category_name" : "category_portuguese", "product_category_name_english" : "category_english"})

def validate(conn):
    print("\n  Validation checks:")

    # Row counts
    for t in ["customers","orders","order_items",
              "payments","products","reviews","sellers"]:
        n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f" {t: <30} {n: >9}")

    # Date range
    r = conn.execute("""
            SELECT min(date(order_purchase_timestamp)),
                   max(date(order_purchase_timestamp))
            from orders
            where order_purchase_timestamp is not null
    """).fetchone()
    print(f"\n Date range : {r[0]}  -> {r[1]}")

    # Status breakdown
    rows = conn.execute("""
        select order_status, count(*) as n
        from orders group by order_status order by n desc
    """).fetchall()
    print("\n Order Statuses: ")

if __name__ == '__main__':
    print('='*55)
    print('Step 1 : Database Setup and Import')
    print('='*55)

    conn = get_connection()

    print("\n  [1/3] Creating tables...")
    create_tables(conn)

    print("\n  [2/3] Importing CSVs...")
    import_all_csvs(conn)

    print("\n  [3/3] Validating...")
    validate(conn)

    conn.close()
    db_mb = os.path.getsize(db_path) / 1024 / 1024
    print(f"\n Database: {db_path} ({db_mb:.1f} MB)")
    print('✅ Step 1 Done')
    print('='*55)


Step 1 : Database Setup and Import

  [1/3] Creating tables...
Dropping table: order_items
Dropping table: payments
Dropping table: reviews
Dropping table: orders
Dropping table: customers
Dropping table: products
Dropping table: sellers
Dropping table: category_translation
✅ Tables and indexes created

  [2/3] Importing CSVs...
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows
✅ {table : <40} {n:>9} rows

  [3/3] Validating...

  Validation checks:
 customers                          99441
 orders                             99441
 order_items                       112650
 payments                          103886
 products                           32951
 reviews                            99224
 sellers                             3095

 Date range : 2016-09-04  -> 2018-10-17

 Order Statuses: 

 Database: C:\Users\bhuvancw\OneDrive\Desktop